# Road Accident Severity Prediction using Machine Learning

## Phase 8 — Hyperparameter Tuning

---

**Project:** Road Accident Severity Prediction using Machine Learning
**Institution:** On-Campus Research Internship, IIIT Vadodara
**Notebook:** `07_Hyperparameter_Tuning.ipynb`
**Phase:** 8 of N — Hyperparameter Tuning
**Input Data:** `Dataset/processed/ml_ready/` (`X_train`, `X_test`, `y_train`, `y_test`)
**Target Variable:** `Accident_Severity`

---

### Objective of this Notebook

Phase 7 trained and compared 8–10 classifiers using default hyperparameters. This
notebook optimizes the three strongest baseline model families — **Random Forest**,
**Extra Trees**, and **Gradient Boosting** — using `RandomizedSearchCV`, then compares
each tuned model directly against its own baseline to quantify the actual improvement
from tuning.

Specifically, this notebook:

1. Loads the prepared `X_train`, `X_test`, `y_train`, `y_test` arrays.
2. Trains fresh baseline models (Random Forest, Extra Trees, Gradient Boosting) using
   default hyperparameters, for a fair, apples-to-apples comparison against the tuned
   models trained later in this same notebook.
3. Defines a `RandomizedSearchCV` search space for each of the three model families.
4. Tunes each model using **Stratified 5-Fold Cross-Validation**, `random_state=42`,
   and **weighted F1 Score** as the scoring metric.
5. Reports the best parameters, best cross-validation score, and training time for
   each tuned model.
6. Evaluates every tuned model on the untouched test set: Accuracy, Precision, Recall,
   F1 Score, Balanced Accuracy, and a confusion matrix.
7. Builds a Baseline-vs-Tuned comparison table and bar charts for every metric.
8. Saves only the **tuned** models to `models/trained/` and the comparison table to
   `reports/tuned_model_results.csv`.

> **Why `RandomizedSearchCV` instead of `GridSearchCV`:** an exhaustive grid search
> over even a modest number of hyperparameters multiplies combinatorially and becomes
> impractical on a dataset of this size; `RandomizedSearchCV` samples a fixed, bounded
> number of combinations, giving strong practical results in a fraction of the time.

> **Metric averaging note:** to keep the tuning objective and the reported test metrics
> consistent, this notebook uses **weighted averaging** throughout (matching the
> weighted F1 scoring metric used to drive the search) — a deliberate, documented
> difference from Phase 7's baseline notebook, which used macro averaging to foreground
> minority-class performance. Baseline models are **trained fresh, in memory, inside
> this notebook** (rather than loaded from `models/trained/`) and evaluated under this
> same weighted convention, so the Baseline-vs-Tuned comparison is always
> apples-to-apples.


---
## Setup — Import Libraries & Configure Environment

**Purpose:** Import the libraries required for hyperparameter tuning, evaluation, and
visualization; configure plotting style; and define all output paths used throughout
this notebook.


In [1]:
# ---- Core Libraries ----
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)

# ---- Environment Configuration ----
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["figure.dpi"] = 100

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
TARGET_COLUMN = "Accident_Severity"
SCORING_METRIC = "f1_weighted"
METRIC_AVERAGE = "weighted"
CV_FOLDS = 5
N_ITER = 20

# ---- Paths ----
ML_READY_DIR = Path("..") / "Dataset" / "processed" / "ml_ready"
MODELS_DIR = Path("..") / "models" / "trained"
FIGURES_DIR = Path("..") / "reports" / "figures" / "hyperparameter_tuning"
TUNED_RESULTS_PATH = Path("..") / "reports" / "tuned_model_results.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TUNED_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename: str) -> Path:
    '''
    Save a matplotlib figure to the hyperparameter tuning figures directory.

    Args:
        fig: The matplotlib Figure object to save.
        filename (str): Filename (including extension) to save the figure as.

    Returns:
        Path: The full path the figure was saved to.
    '''
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, bbox_inches="tight", dpi=150)
    print(f"Figure saved to: {output_path.resolve()}")
    return output_path


def sanitize_filename(name: str) -> str:
    '''
    Convert a human-readable model name into a safe, lowercase,
    underscore-separated filename stem.

    Args:
        name (str): Human-readable model name (e.g., "Random Forest").

    Returns:
        str: A filesystem-safe filename stem (e.g., "random_forest").
    '''
    return name.lower().replace(" ", "_").replace("-", "_")


print("Libraries imported and environment configured successfully.")
print(f"Scoring metric for tuning : {SCORING_METRIC}")
print(f"Cross-validation strategy : Stratified {CV_FOLDS}-Fold, random_state={RANDOM_STATE}")
print(f"RandomizedSearchCV budget : {N_ITER} sampled combinations per model")


Libraries imported and environment configured successfully.
Scoring metric for tuning : f1_weighted
Cross-validation strategy : Stratified 5-Fold, random_state=42
RandomizedSearchCV budget : 20 sampled combinations per model


**Interpretation**

- Every required library (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`,
  `joblib`) is imported, plus `scipy.stats` for continuous hyperparameter
  distributions (`randint`, `uniform`) used in the search spaces below.
- All output directories are created up front with `mkdir(parents=True,
  exist_ok=True)`, and `save_figure()`/`sanitize_filename()` are reused in the same
  form established in Phase 7, for consistency across notebooks.


---
## 1. Load ML-Ready Data

**Purpose:** Load the prepared `X_train`, `X_test`, `y_train`, and `y_test` arrays
from `Dataset/processed/ml_ready/`, using robust exception handling consistent with
prior notebooks.


In [2]:
def load_ml_ready_csv(filename: str) -> pd.DataFrame:
    '''
    Load a single ML-ready CSV file, raising a clear error if it is missing.

    Args:
        filename (str): Name of the CSV file to load, relative to ML_READY_DIR.

    Returns:
        pd.DataFrame: The loaded dataframe.
    '''
    file_path = ML_READY_DIR / filename
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path.resolve()}. "
            f"Please run 05_Feature_Selection_and_Data_Preparation.ipynb first."
        )
    return pd.read_csv(file_path)


try:
    X_train = load_ml_ready_csv("X_train.csv")
    X_test = load_ml_ready_csv("X_test.csv")
    y_train = load_ml_ready_csv("y_train.csv")[TARGET_COLUMN]
    y_test = load_ml_ready_csv("y_test.csv")[TARGET_COLUMN]

    print("ML-ready data loaded successfully:")
    print(f"  X_train : {X_train.shape}")
    print(f"  X_test  : {X_test.shape}")
    print(f"  y_train : {y_train.shape}")
    print(f"  y_test  : {y_test.shape}")
except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
    print(f"[ERROR] {error}")
    raise

CLASS_LABELS = sorted(y_test.unique().tolist())
NUMBER_OF_FEATURES = X_train.shape[1]
print(f"\nClass Labels     : {CLASS_LABELS}")
print(f"Feature Count    : {NUMBER_OF_FEATURES}")


ML-ready data loaded successfully:
  X_train : (2172752, 154)
  X_test  : (543188, 154)
  y_train : (2172752,)
  y_test  : (543188,)

Class Labels     : ['Fatal', 'Serious', 'Slight']
Feature Count    : 154


**Interpretation**

- This is the exact same prepared dataset used in Phase 7: a SMOTENC-balanced
  training set and an untouched, realistically imbalanced test set — ensuring tuning
  results and baseline comparisons in this notebook rest on identical data.


---
## 2. Verify Data

**Purpose:** Re-confirm the target distributions of the training and test sets
before tuning any model.


In [3]:
print("Training Target Distribution (%):")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print()
print("Testing Target Distribution (%):")
print(y_test.value_counts(normalize=True).mul(100).round(2))


Training Target Distribution (%):
Accident_Severity
Slight    85.2700
Serious   13.4300
Fatal      1.3100
Name: proportion, dtype: float64

Testing Target Distribution (%):
Accident_Severity
Slight    85.2700
Serious   13.4300
Fatal      1.3100
Name: proportion, dtype: float64


**Interpretation**

- As in Phase 7, the training set is expected to be balanced (SMOTENC-resampled) while
  the test set retains the real-world class imbalance, so every metric reported later
  in this notebook reflects genuine, realistic performance.


---
## 3. Hyperparameter Search Spaces

**Purpose:** Define the `RandomizedSearchCV` parameter distributions for each of the
three models being tuned, exactly matching the parameters specified for this phase.


In [4]:
RANDOM_FOREST_PARAM_DISTRIBUTIONS = {
    "n_estimators": randint(100, 500),
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
}

EXTRA_TREES_PARAM_DISTRIBUTIONS = {
    "n_estimators": randint(100, 500),
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
}

GRADIENT_BOOSTING_PARAM_DISTRIBUTIONS = {
    "n_estimators": randint(100, 400),
    "learning_rate": uniform(0.01, 0.29),  # samples in [0.01, 0.30)
    "max_depth": randint(2, 7),
    "subsample": uniform(0.6, 0.4),  # samples in [0.6, 1.0)
}

print("Random Forest search space:")
for param, values in RANDOM_FOREST_PARAM_DISTRIBUTIONS.items():
    print(f"  - {param}: {values}")

print("\nExtra Trees search space:")
for param, values in EXTRA_TREES_PARAM_DISTRIBUTIONS.items():
    print(f"  - {param}: {values}")

print("\nGradient Boosting search space:")
for param, values in GRADIENT_BOOSTING_PARAM_DISTRIBUTIONS.items():
    print(f"  - {param}: {values}")


Random Forest search space:
  - n_estimators: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000022F55BA4590>
  - max_depth: [None, 5, 10, 15, 20, 30]
  - min_samples_split: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000022F7FB020D0>
  - min_samples_leaf: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000022F7FB02350>
  - max_features: ['sqrt', 'log2', None]

Extra Trees search space:
  - n_estimators: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000022F7FA8B230>
  - max_depth: [None, 5, 10, 15, 20, 30]
  - min_samples_split: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000022F7FA8B490>
  - min_samples_leaf: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000022F7FB29C70>

Gradient Boosting search space:
  - n_estimators: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000022F7FB88490>
  - learning_rate: <scipy.stats._distn_infrastructure.rv_

**Interpretation**

- Each search space matches exactly the parameters specified for this phase: Random
  Forest and Extra Trees search `n_estimators`, `max_depth`, `min_samples_split`, and
  `min_samples_leaf` (with Random Forest additionally searching `max_features`, per
  its specification); Gradient Boosting searches `n_estimators`, `learning_rate`,
  `max_depth`, and `subsample`.
- Continuous-valued parameters (`learning_rate`, `subsample`) use `scipy.stats.uniform`
  distributions rather than a fixed discrete list, letting `RandomizedSearchCV` sample
  genuinely random values across the specified range — one of the practical advantages
  of randomized over grid search.


---
## 4. Reusable Tuning & Evaluation Functions

**Purpose:** Define reusable functions for running `RandomizedSearchCV`, evaluating
any fitted model on the test set with weighted metrics, and plotting/saving a
confusion matrix — so the same logic drives baseline evaluation, all three tuning
runs, and the final comparison, without duplication.


In [5]:
def plot_and_save_confusion_matrix(cm: np.ndarray, class_labels: list, model_name: str) -> Path:
    '''
    Plot a confusion matrix as an annotated heatmap and save it to the
    hyperparameter tuning figures directory.

    Args:
        cm (np.ndarray): The confusion matrix (rows = true, columns = predicted).
        class_labels (list): Ordered class labels corresponding to the matrix axes.
        model_name (str): Human-readable model name, used in the title/filename.

    Returns:
        Path: The full path the figure was saved to.
    '''
    fig, ax = plt.subplots(figsize=(7, 6))

    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_labels, yticklabels=class_labels, cbar=True, ax=ax,
    )

    ax.set_title(f"Confusion Matrix — {model_name}", fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    plt.tight_layout()
    filename = f"confusion_matrix_{sanitize_filename(model_name)}.png"
    saved_path = save_figure(fig, filename)
    plt.show()

    return saved_path


def evaluate_model_on_test(
    model, model_name: str, X_test: pd.DataFrame, y_test: pd.Series, class_labels: list
) -> dict:
    '''
    Evaluate a fitted model on the test set using weighted-averaged
    metrics, plot and save its confusion matrix, and time the prediction.

    Args:
        model: A fitted scikit-learn-compatible classifier.
        model_name (str): Human-readable name of the model (used in titles/filenames).
        X_test (pd.DataFrame): Test features.
        y_test (pd.Series): Test target.
        class_labels (list): Ordered class labels for the confusion matrix.

    Returns:
        dict: A row of test-set metrics and prediction timing for this model.
    '''
    predict_start = time.time()
    y_pred = model.predict(X_test)
    prediction_time = time.time() - predict_start

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average=METRIC_AVERAGE, zero_division=0)
    recall = recall_score(y_test, y_pred, average=METRIC_AVERAGE, zero_division=0)
    f1 = f1_score(y_test, y_pred, average=METRIC_AVERAGE, zero_division=0)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    plot_and_save_confusion_matrix(cm, class_labels, model_name)

    print(
        f"  Accuracy={accuracy:.4f}  Precision={precision:.4f}  Recall={recall:.4f}  "
        f"F1={f1:.4f}  Balanced Accuracy={balanced_accuracy:.4f}  "
        f"Predict Time={prediction_time:.4f}s"
    )

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Balanced Accuracy": balanced_accuracy,
        "Prediction Time (s)": prediction_time,
    }


def tune_model_with_randomized_search(
    estimator, param_distributions: dict, model_name: str,
    X_train: pd.DataFrame, y_train: pd.Series,
) -> dict:
    '''
    Run RandomizedSearchCV for a single estimator using Stratified 5-Fold
    cross-validation and weighted F1 scoring, and report the best
    parameters, best CV score, and total search training time.

    Args:
        estimator: An unfitted scikit-learn-compatible classifier.
        param_distributions (dict): Hyperparameter distributions to sample from.
        model_name (str): Human-readable name of the model.
        X_train (pd.DataFrame): Training features.
        y_train (pd.Series): Training target.

    Returns:
        dict: Contains the fitted 'best_estimator', 'best_params', 'best_cv_score',
              and 'training_time' for this tuning run.
    '''
    print(f"Tuning '{model_name}' with RandomizedSearchCV ({N_ITER} candidates, "
          f"{CV_FOLDS}-fold stratified CV, scoring='{SCORING_METRIC}')...")

    cv_strategy = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=N_ITER,
        scoring=SCORING_METRIC,
        cv=cv_strategy,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        refit=True,
    )

    train_start = time.time()
    search.fit(X_train, y_train)
    training_time = time.time() - train_start

    print(f"  Best CV Score ({SCORING_METRIC}): {search.best_score_:.4f}")
    print(f"  Training Time: {training_time:.2f} seconds")
    print(f"  Best Parameters: {search.best_params_}")

    return {
        "best_estimator": search.best_estimator_,
        "best_params": search.best_params_,
        "best_cv_score": search.best_score_,
        "training_time": training_time,
    }


print("Reusable tuning, evaluation, and plotting functions defined.")


Reusable tuning, evaluation, and plotting functions defined.


**Interpretation**

- `tune_model_with_randomized_search()` centralizes the entire search configuration —
  cross-validation strategy, scoring metric, iteration budget, random state — in one
  place, so every model is tuned under identical, fair conditions and any future change
  to the search protocol only needs to be made once.
- `evaluate_model_on_test()` is intentionally identical in structure to Phase 7's
  evaluation function, but uses **weighted** averaging throughout, matching this
  notebook's tuning objective — this is reused for the baseline models below as well
  as every tuned model, guaranteeing a fair, consistent comparison.


---
## 5. Fresh Baseline Model Training & Evaluation

**Purpose:** Train fresh baseline models — Random Forest, Extra Trees, and Gradient
Boosting — directly in this notebook, using default hyperparameters, then evaluate
each on the test set using this notebook's **weighted**-average metric convention.

> **Note on this design choice:** an earlier version of this notebook loaded
> previously saved baseline models from `models/trained/` via `joblib.load()`. On
> large datasets, holding a full-size, already-trained ensemble model in memory
> *alongside* the full training/test arrays and the models about to be tuned can
> exceed available memory, causing a `MemoryError`. This notebook avoids that
> entirely by **training its own lightweight baseline models fresh, in memory**, and
> never loading any previously saved model file — removing that dependency, and that
> failure mode, completely.


In [ ]:
BASELINE_ESTIMATORS = {
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

baseline_models = {}
baseline_results = {}
baseline_training_times = {}

for model_name, estimator in BASELINE_ESTIMATORS.items():
    print(f"\n--- Training fresh baseline: {model_name} ---")

    train_start = time.time()
    estimator.fit(X_train, y_train)
    training_time = time.time() - train_start

    print(f"  Training Time: {training_time:.2f} seconds")

    baseline_models[model_name] = estimator
    baseline_training_times[model_name] = training_time
    baseline_results[model_name] = evaluate_model_on_test(
        estimator, f"Baseline {model_name}", X_test, y_test, CLASS_LABELS
    )



--- Training fresh baseline: Random Forest ---


**Interpretation**

- Every baseline model is trained fresh, right here, with only `random_state` set
  (default hyperparameters otherwise) — the exact same convention used for the
  baseline comparison in Phase 7, just computed locally rather than loaded from disk.
- Because these baseline models are trained fresh and **not** persisted to
  `models/trained/` (per this phase's scope — only the tuned models are saved, in
  Section 10), there is no risk of them being confused with either the Phase 7
  baseline artifacts or the tuned models saved later in this notebook.
- This also means the notebook has **no dependency at all** on `models/trained/`
  already containing anything before this notebook runs — it is fully self-contained
  and standalone.


---
## 6. Hyperparameter Tuning — Random Forest


In [ ]:
random_forest_tuning = tune_model_with_randomized_search(
    RandomForestClassifier(random_state=RANDOM_STATE),
    RANDOM_FOREST_PARAM_DISTRIBUTIONS,
    "Random Forest",
    X_train, y_train,
)

tuned_random_forest = random_forest_tuning["best_estimator"]


In [ ]:
print("--- Random Forest: Tuned Model — Test Set Evaluation ---")
random_forest_tuned_results = evaluate_model_on_test(
    tuned_random_forest, "Tuned Random Forest", X_test, y_test, CLASS_LABELS
)


**Interpretation**

- The best parameters and cross-validation score above reflect the single
  best-performing combination out of the `N_ITER` randomly sampled candidates,
  evaluated under Stratified 5-Fold CV with weighted F1 scoring.
- The test-set evaluation immediately above uses the same held-out test set as every
  other model in this project, so its metrics are directly comparable to the Random
  Forest baseline evaluated in Section 5.


---
## 7. Hyperparameter Tuning — Extra Trees


In [ ]:
extra_trees_tuning = tune_model_with_randomized_search(
    ExtraTreesClassifier(random_state=RANDOM_STATE),
    EXTRA_TREES_PARAM_DISTRIBUTIONS,
    "Extra Trees",
    X_train, y_train,
)

tuned_extra_trees = extra_trees_tuning["best_estimator"]


In [ ]:
print("--- Extra Trees: Tuned Model — Test Set Evaluation ---")
extra_trees_tuned_results = evaluate_model_on_test(
    tuned_extra_trees, "Tuned Extra Trees", X_test, y_test, CLASS_LABELS
)


**Interpretation**

- Extra Trees differs from Random Forest by selecting split thresholds randomly rather
  than optimally, which typically trades a small amount of per-tree accuracy for lower
  variance and faster training — the tuned parameters above reflect whichever
  combination best balanced this trade-off under cross-validation.


---
## 8. Hyperparameter Tuning — Gradient Boosting


In [ ]:
gradient_boosting_tuning = tune_model_with_randomized_search(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    GRADIENT_BOOSTING_PARAM_DISTRIBUTIONS,
    "Gradient Boosting",
    X_train, y_train,
)

tuned_gradient_boosting = gradient_boosting_tuning["best_estimator"]


In [ ]:
print("--- Gradient Boosting: Tuned Model — Test Set Evaluation ---")
gradient_boosting_tuned_results = evaluate_model_on_test(
    tuned_gradient_boosting, "Tuned Gradient Boosting", X_test, y_test, CLASS_LABELS
)


**Interpretation**

- Gradient Boosting builds trees sequentially, each correcting the previous ensemble's
  errors; `learning_rate` and `n_estimators` jointly control the bias-variance
  trade-off (many trees at a low learning rate vs. fewer trees at a higher one), while
  `subsample` introduces stochasticity that can reduce overfitting — the tuned values
  above reflect the best-performing balance found under cross-validation.
- **Runtime note:** Gradient Boosting trains trees sequentially and cannot parallelize
  within a single fit, making it the slowest of the three models to tune; this is an
  inherent algorithmic characteristic, not a notebook inefficiency.


---
## 9. Model Comparison — Baseline vs Tuned

**Purpose:** Consolidate every baseline and tuned model's test-set metrics into a
single comparison table and a set of bar charts, to directly quantify the improvement
(or lack thereof) achieved by hyperparameter tuning for each model family.


In [ ]:
tuned_results = {
    "Random Forest": random_forest_tuned_results,
    "Extra Trees": extra_trees_tuned_results,
    "Gradient Boosting": gradient_boosting_tuned_results,
}

tuned_params = {
    "Random Forest": random_forest_tuning["best_params"],
    "Extra Trees": extra_trees_tuning["best_params"],
    "Gradient Boosting": gradient_boosting_tuning["best_params"],
}

tuned_cv_scores = {
    "Random Forest": random_forest_tuning["best_cv_score"],
    "Extra Trees": extra_trees_tuning["best_cv_score"],
    "Gradient Boosting": gradient_boosting_tuning["best_cv_score"],
}

tuned_training_times = {
    "Random Forest": random_forest_tuning["training_time"],
    "Extra Trees": extra_trees_tuning["training_time"],
    "Gradient Boosting": gradient_boosting_tuning["training_time"],
}

comparison_rows = []
for model_name in ["Random Forest", "Extra Trees", "Gradient Boosting"]:
    baseline_row = {
        "Model": model_name,
        "Stage": "Baseline",
        **baseline_results[model_name],
        "Training Time (s)": baseline_training_times[model_name],
    }
    tuned_row = {
        "Model": model_name,
        "Stage": "Tuned",
        **tuned_results[model_name],
        "Best CV Score (f1_weighted)": tuned_cv_scores[model_name],
        "Best Parameters": str(tuned_params[model_name]),
        "Training Time (s)": tuned_training_times[model_name],
    }
    comparison_rows.append(baseline_row)
    comparison_rows.append(tuned_row)

comparison_df = pd.DataFrame(comparison_rows)

# Baseline rows have no tuning-specific values (best CV score, best
# parameters) by definition — fill these explicitly with "N/A" rather than
# leaving a bare NaN, so the saved CSV clearly communicates "not
# applicable" rather than looking like missing data. Training Time is now
# populated for both stages, since baseline models are trained fresh in
# this notebook rather than loaded pre-trained from disk.
for column in ["Best CV Score (f1_weighted)", "Best Parameters"]:
    comparison_df[column] = comparison_df[column].where(comparison_df[column].notna(), "N/A")

comparison_df


In [ ]:
def plot_baseline_vs_tuned(comparison_df: pd.DataFrame, metric: str) -> Path:
    '''
    Generate and save a grouped bar chart comparing Baseline vs Tuned
    performance on a given metric, across all three tuned model families.

    Args:
        comparison_df (pd.DataFrame): The full baseline/tuned comparison table.
        metric (str): Name of the metric column to plot.

    Returns:
        Path: The full path the figure was saved to.
    '''
    pivot_df = comparison_df.pivot(index="Model", columns="Stage", values=metric)
    pivot_df = pivot_df[["Baseline", "Tuned"]]

    fig, ax = plt.subplots(figsize=(10, 6))
    x_positions = np.arange(len(pivot_df))
    bar_width = 0.35

    baseline_bars = ax.bar(
        x_positions - bar_width / 2, pivot_df["Baseline"], bar_width,
        label="Baseline", color="#8C8C8C",
    )
    tuned_bars = ax.bar(
        x_positions + bar_width / 2, pivot_df["Tuned"], bar_width,
        label="Tuned", color="#4C72B0",
    )

    for bars in (baseline_bars, tuned_bars):
        for bar in bars:
            height = bar.get_height()
            ax.annotate(
                f"{height:.3f}",
                (bar.get_x() + bar.get_width() / 2, height),
                ha="center", va="bottom", fontsize=9, xytext=(0, 3),
                textcoords="offset points",
            )

    ax.set_title(f"Baseline vs Tuned — {metric}", fontweight="bold")
    ax.set_xlabel("Model")
    ax.set_ylabel(metric)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(pivot_df.index)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    filename = f"baseline_vs_tuned_{sanitize_filename(metric)}.png"
    saved_path = save_figure(fig, filename)
    plt.show()

    return saved_path


for metric_name in ["Accuracy", "Precision", "Recall", "F1 Score", "Balanced Accuracy"]:
    plot_baseline_vs_tuned(comparison_df, metric_name)


In [ ]:
improvement_rows = []
for model_name in ["Random Forest", "Extra Trees", "Gradient Boosting"]:
    baseline_f1 = baseline_results[model_name]["F1 Score"]
    tuned_f1 = tuned_results[model_name]["F1 Score"]
    baseline_accuracy = baseline_results[model_name]["Accuracy"]
    tuned_accuracy = tuned_results[model_name]["Accuracy"]

    improvement_rows.append(
        {
            "Model": model_name,
            "Baseline F1": baseline_f1,
            "Tuned F1": tuned_f1,
            "F1 Improvement": tuned_f1 - baseline_f1,
            "Baseline Accuracy": baseline_accuracy,
            "Tuned Accuracy": tuned_accuracy,
            "Accuracy Improvement": tuned_accuracy - baseline_accuracy,
        }
    )

improvement_df = pd.DataFrame(improvement_rows).sort_values("F1 Improvement", ascending=False)
improvement_df


**Interpretation**

- **What the charts show:** Each chart directly pairs a model's baseline (default
  hyperparameter) performance against its tuned counterpart on one metric, making any
  improvement — or regression — immediately visible.
- **Why some models may improve more than others:** tree-ensemble methods with strong
  default hyperparameters (like Random Forest and Extra Trees) sometimes show only
  modest gains from tuning, since their defaults are already reasonably well-suited to
  many tabular problems; Gradient Boosting, whose defaults are more conservative
  (fewer, shallower trees), often has more room for improvement from tuning
  `n_estimators`, `learning_rate`, and `max_depth` jointly.
- **Impact on the project:** the model with the highest **Tuned F1 Score** in this
  comparison should be considered the leading candidate going into any final model
  selection or deployment discussion in a later phase.


---
## 10. Save Tuned Models

**Purpose:** Persist every tuned model to `models/trained/` using `joblib`, under
clearly distinguished `best_*` filenames so they are never confused with the Phase 7
baseline model files of the same family.


In [ ]:
tuned_models_to_save = {
    "best_random_forest.pkl": tuned_random_forest,
    "best_extra_trees.pkl": tuned_extra_trees,
    "best_gradient_boosting.pkl": tuned_gradient_boosting,
}

for filename, model in tuned_models_to_save.items():
    model_path = MODELS_DIR / filename
    joblib.dump(model, model_path)
    print(f"Saved: {model_path.resolve()}")


**Interpretation**

- Only the **tuned** models are persisted here — the fresh baseline models trained in
  Section 5 exist only in memory for comparison purposes and are intentionally never
  saved, consistent with this phase's scope and with avoiding the memory pressure of
  accumulating extra large model artifacts on disk.
- Each tuned model is saved under a `best_*.pkl` filename, distinct from any Phase 7
  baseline `*.pkl` files that may already exist in `models/trained/`, so both remain
  available for later phases (e.g., final model selection, deployment, or further
  analysis) without ambiguity about which is which.


---
## 11. Save Results

**Purpose:** Persist the full Baseline-vs-Tuned comparison table to
`reports/tuned_model_results.csv` for reference in later phases and the final
research report.


In [ ]:
try:
    comparison_df.to_csv(TUNED_RESULTS_PATH, index=False)
    print(f"Tuned model comparison results saved to: {TUNED_RESULTS_PATH.resolve()}")
except OSError as save_error:
    print(f"[ERROR] Failed to save tuned model comparison results: {save_error}")
    raise


**Interpretation**

- The saved CSV contains both the `Baseline` and `Tuned` rows for all three model
  families, including the tuned models' best cross-validation score, best parameters,
  and training time — a complete, self-contained record of this phase's results.


---
## 12. Final Summary

**Purpose:** Print a consolidated final summary identifying the best tuned model and
the magnitude of improvement achieved through hyperparameter tuning.


In [ ]:
best_tuned_row = improvement_df.iloc[0]
best_tuned_model_name = best_tuned_row["Model"]

print("=" * 60)
print("HYPERPARAMETER TUNING — FINAL SUMMARY")
print("=" * 60)

print(f"\nBest Tuned Model       : {best_tuned_model_name}")
print(f"Tuned F1 Score          : {best_tuned_row['Tuned F1']:.4f}")
print(f"Baseline F1 Score       : {best_tuned_row['Baseline F1']:.4f}")
print(f"F1 Improvement          : {best_tuned_row['F1 Improvement']:+.4f}")
print(f"Tuned Accuracy          : {best_tuned_row['Tuned Accuracy']:.4f}")
print(f"Accuracy Improvement    : {best_tuned_row['Accuracy Improvement']:+.4f}")
print(f"Best CV Score (weighted F1) : {tuned_cv_scores[best_tuned_model_name]:.4f}")
print(f"Training Time            : {tuned_training_times[best_tuned_model_name]:.2f} seconds")
print(f"Number of Features        : {NUMBER_OF_FEATURES}")

print("\n" + "=" * 60)
print(f"Tuned models saved to      : {MODELS_DIR.resolve()}")
print(f"Comparison figures saved to: {FIGURES_DIR.resolve()}")
print(f"Comparison table saved to  : {TUNED_RESULTS_PATH.resolve()}")
print("=" * 60)


**Interpretation**

- This summary confirms the phase's core deliverables are complete: all three model
  families tuned via `RandomizedSearchCV` under Stratified 5-Fold CV with weighted F1
  scoring, evaluated consistently against their baselines, saved, and compared — with
  the best-performing tuned model identified programmatically.

### Next Steps

The next notebook in the project roadmap will take the best tuned model identified
here forward for final model selection, deeper error analysis, and/or deployment —
none of which is performed in this hyperparameter-tuning-only notebook.
